# Shape demo: load the demo Parquet files into Delta tables

Run once after uploading the demo data (produced by `demo/make_data.py`) to the
lakehouse `Files/demo/` folder. Every `*.parquet` file (one folder level down is also
accepted) becomes a Delta table named after the file, for example
`Files/demo/day1/orders.parquet` becomes the table `orders_day1`, and
`Files/demo/orders.parquet` becomes `orders`.

Attach the default lakehouse before running. Uses the preinstalled `deltalake` package.


In [ ]:
import os
from pathlib import Path

import pyarrow.parquet as pq
from deltalake import write_deltalake

LAKEHOUSE = "/lakehouse/default"
DEMO_DIR = Path(LAKEHOUSE) / "Files" / "demo"
TABLES_DIR = Path(LAKEHOUSE) / "Tables"


In [ ]:
def table_name_for(parquet_file: Path) -> str:
    """Files/demo/<sub>/<name>.parquet -> <name>_<sub>; Files/demo/<name>.parquet -> <name>."""
    rel = parquet_file.relative_to(DEMO_DIR)
    parts = [p.lower().replace("-", "_").replace(" ", "_") for p in rel.parts]
    stem = Path(parts[-1]).stem
    return "_".join([stem, *parts[:-1]]) if len(parts) > 1 else stem


files = sorted(DEMO_DIR.rglob("*.parquet"))
if not files:
    raise FileNotFoundError(
        f"No .parquet files under {DEMO_DIR}. Upload the demo data to Files/demo/ first."
    )


In [ ]:
created = []
for f in files:
    name = table_name_for(f)
    table = pq.read_table(f)
    write_deltalake(str(TABLES_DIR / name), table, mode="overwrite")
    created.append({"table": name, "rows": table.num_rows, "source": str(f.relative_to(DEMO_DIR))})

for row in created:
    print(f"{row['table']:<32} {row['rows']:>10,} rows   <- {row['source']}")
